# 📊 Recommandations et restitution

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 2 · 13h30** · Préparation de la restitution évaluée

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Formuler des recommandations opérationnelles structurées
- Intégrer les indicateurs ESG, les limites et les retours utilisateurs
- Préparer 15 minutes de restitution adaptées à chaque interlocuteur

In [1]:
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

✅ Configuration prête : chaque fichier de données se télécharge au premier usage


In [2]:
# Secteur traité : Crédit Azur (banque) — autres choix : "commerce", "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Crédit Azur · 20477 lignes, 20 colonnes


## 1. Les chiffres clés de votre étude
Rassemblez dans un tableau les cinq chiffres que vous citerez à l'oral.

In [3]:
from fairlearn.metrics import demographic_parity_ratio
df = nettoyage_minimal(brut, cfg)
ev = pd.Timestamp(cfg["evenement"][0])
avant, apres = df[df[cfg["date"]] < ev], df[df[cfg["date"]] >= ev]
dec = cfg["decision"]
tx = df.groupby("tranche_age")[dec].mean()
ratio_age = demographic_parity_ratio(df[dec], df[dec], sensitive_features=df["tranche_age"])
pnb_m = df.set_index(cfg["date"])[cfg["ca"]].resample("MS").sum()
clients = df.drop_duplicates("client_id")
part_65 = (clients["tranche_age"] == "65+").mean()
acc = df[df[dec] == 1]

chiffres = pd.DataFrame([
    ("Taux d'accord avant / après le score V2", f"{avant[dec].mean():.0%} → {apres[dec].mean():.0%}", f"{(apres[dec].mean() - avant[dec].mean()) * 100:+.0f} points à partir de {ev:%m/%Y}"),
    ("PNB mensuel moyen avant / après", f"{pnb_m[pnb_m.index < ev].mean() / 1e3:,.0f} k€ → {pnb_m[pnb_m.index >= ev].mean() / 1e3:,.0f} k€",
     f"{pnb_m[pnb_m.index >= ev].mean() / pnb_m[pnb_m.index < ev].mean() - 1:+.0%}"),
    ("Taux d'accord des 18-24 ans vs 55-64 ans", f"{tx['18-24']:.0%} vs {tx['55-64']:.0%}", f"ratio de parité d'âge {ratio_age:.2f} (seuil 0,8)"),
    ("Taux d'accord des 18-24 ans, avant → après", f"{avant[avant.tranche_age == '18-24'][dec].mean():.0%} → {apres[apres.tranche_age == '18-24'][dec].mean():.0%}", "l'écart d'âge se creuse avec le score V2"),
    ("Remboursement sans incident des 18-24 ans accordés", f"{acc[acc.tranche_age == '18-24'][cfg['resultat']].mean():.1%}",
     f"contre {acc[acc.tranche_age == '55-64'][cfg['resultat']].mean():.1%} pour les 55-64 ans : écart de risque observé faible"),
], columns=["Indicateur", "Valeur", "Lecture"])
pd.set_option("display.max_colwidth", 90)
chiffres

,Indicateur,Valeur,Lecture
0,Taux d'accord avant / après le score V2,76% → 57%,-19 points à partir de 03/2024
1,PNB mensuel moyen avant / après,343 k€ → 266 k€,-23%
2,Taux d'accord des 18-24 ans vs 55-64 ans,18% vs 87%,"ratio de parité d'âge 0.21 (seuil 0,8)"
3,"Taux d'accord des 18-24 ans, avant → après",36% → 8%,l'écart d'âge se creuse avec le score V2
4,Remboursement sans incident des 18-24 ans accordés,95.1%,contre 98.2% pour les 55-64 ans : écart de risque observé faible


## 2. Vos recommandations
Cinq recommandations, classées par impact, chacune sur le même modèle.

| Constat chiffré | Recommandation | Impact attendu | Indicateur de suivi | Limite ou condition |
|---|---|---|---|---|
| Le taux d'accord est passé de **76 % à 57 %** (−19 points) en mars 2024, le PNB mensuel de **343 k€ à 266 k€** (−23 %) alors que les demandes augmentent de 7 % | **1. Recalibrer le seuil du score V2** par un test sur 10 % des dossiers dont le score est proche du seuil (entre 0,6 et 0,7), avec décision de généralisation au bout de 2 trimestres | Remonter vers 68 % d'accord : environ **+50 k€ de PNB par mois** (≈ 4,7 k€ par point d'accord) | Taux d'accord mensuel ; taux de remboursement sans incident des crédits accordés (rester ≥ 97 %) | Le PNB n'est pas la marge : il faut y retirer le coût du risque, qui n'est pas dans les données |
| Les 18-24 ans passent de **36 % à 8 %** d'accord après V2 ; ratio de parité d'âge **0,38 → 0,09** (seuil 0,8) ; à profil de risque égal, l'interaction âge × V2 divise leurs chances par 7 (rapport de cotes 0,14) | **2. Auditer le score V2 sur les 18-34 ans** (poids de l'ancienneté et du contrat) et ajouter un **contrôle humain** sur les dossiers 18-24 ans à score intermédiaire | Ratio de parité d'âge **au-dessus de 0,5 en 6 mois** (0,8 serait irréaliste vu les différences de profil) | Ratio de parité d'âge et taux d'accord 18-24 ans, chaque trimestre | Une part de l'écart reflète des critères légitimes (45 % de CDD, ancienneté médiane de 0,9 an) |
| Les jeunes **acceptés** remboursent à **95,1 %** contre 98,4 % pour les 65 ans et plus, pour un taux d'accord de 18 % contre 87 % ; le remboursement des refusés est inconnu | **3. Lancer un pilote « jeune actif »** (montant plafonné, durée courte, garant) et accepter à titre d'essai un échantillon aléatoire de refus proches du seuil, pour mesurer leur vrai risque | Mesurer le risque réel des refusés ; récupérer une partie du PNB des 25-34 ans (taux d'accord 60 % → 34 %) | Taux d'incident de la cohorte pilote comparé à un groupe témoin (objectif : moins de 5 %) | 327 jeunes acceptés seulement ; l'essai a un coût de risque borné par le plafond de montants |
| Le score pèse très fort sur la décision (Spearman **0,83**) et la disparité d'âge s'est creusée sans que rien ne la signale ; l'évaluation de solvabilité est un usage **à haut risque** (IA Act) | **4. Mettre en place un suivi trimestriel** de la parité (âge, genre, région) et de la représentativité, avec seuil d'alerte à 0,8 et dossier documenté (données, tests, décisions) | Détecter une dérive dans le trimestre et disposer du dossier de conformité | Ratios de parité par variable sensible ; nombre de décisions revues par un humain | Une disparité déclenche une enquête, pas une conclusion ; les groupes de moins de 300 demandes (Outre-mer) sont trop fragiles |
| Les **65 ans et plus** sont **6,9 %** des clients contre 26 % de la population (ratio 0,26) alors que **81 %** de leurs demandes sont acceptées ; l'Outre-mer est à 1,1 % contre 3 % | **5. Étudier l'accès à la demande** des 65 ans et plus et de l'Outre-mer (offre hors ligne, courtiers locaux) avant tout objectif commercial | Mieux couvrir ces publics ; le problème est en amont de la décision | Part des 65 ans et plus et de l'Outre-mer parmi les clients (ratio de représentativité) | La référence est la population nationale, pas les demandeurs de crédit : l'écart est en partie normal |

## 3. Les indicateurs ESG
Quels indicateurs environnementaux, sociaux ou de gouvernance suivre, au vu de votre analyse ?

- **Social (inclusion financière)** : taux d'accord des 18-24 et 25-34 ans (aujourd'hui 8 % et 34 % depuis V2) ; ratio de parité par âge, genre et région (seuil 0,8) ; part des 65 ans et plus et de l'Outre-mer parmi les clients ; nombre de réclamations et de recours des jeunes actifs, par trimestre (le brief signale leurs plaintes ; ces retours ne sont pas dans les données).
- **Gouvernance** : part des décisions de score revues par un humain ; dossier documenté du score (données, tests d'équité, mesures correctives) conforme à l'usage à haut risque de l'IA Act ; délai entre un changement de score et la détection d'une dérive (ici : rupture visible dès mars 2024) ; revue annuelle du score.
- **Environnement** : les données n'en contiennent pas. Les crédits « travaux » (36 % du PNB) pourraient servir de base à un suivi de la part de rénovation énergétique et de véhicules propres dans le crédit auto (48 % du PNB), **à collecter** : c'est un manque à signaler, pas un résultat.
- **Retours utilisateurs** : à recueillir (réclamations, motifs de refus contestés, enquête auprès des 18-34 ans) pour confirmer ou nuancer les chiffres.

## 4. Adapter le discours
| Interlocuteur | Ce qu'il veut savoir | Ce que vous lui montrez |
|---|---|---|
| Direction métier | Que faire lundi pour retrouver du volume ? | Les recommandations 1, 3 et 5, leurs indicateurs (taux d'accord, remboursement, part des clients) et le calendrier du test de seuil sur 2 trimestres |
| Décideurs (comité de direction) | Combien perdons-nous, et quel risque prenons-nous ? | Trois chiffres : −19 points de taux d'accord, −23 % de PNB (−78 k€ par mois), ratio de parité d'âge de 0,09 ; la courbe avec le score V2 annoté ; la décision attendue : valider le test de seuil et l'audit du score |
| Conformité (DPO, référent éthique) | Est-ce équitable, et est-ce documenté ? | Représentativité et parité par âge, genre, région ; la comparaison à profil de risque égal et dans le temps ; les limites (sélection, référence, effectifs) ; les mesures correctives (contrôle humain, suivi trimestriel, dossier IA Act). Message clair : « sur 100 demandes de 18-24 ans, 8 sont acceptées depuis V2, contre 83 pour les 55-64 ans » |

## 5. Le plan des 15 minutes
| Temps | Contenu |
|---|---|
| 1 min | Le contexte et la question du commanditaire |
| 3 min | Ce que disent les données : tendance, saisonnalité, rupture |
| 3 min | Les segments qui comptent |
| 3 min | Les biais détectés et leurs limites |
| 4 min | Les recommandations et leurs indicateurs |
| 1 min | La décision attendue |

Ensuite, 5 minutes de questions. Chaque membre du groupe prend la parole.

## ✅ Avant de passer
- [ ] Le dashboard HTML s'ouvre sur le poste de présentation.
- [ ] Chaque recommandation a un chiffre, un indicateur et une limite.
- [ ] Les biais sont présentés sans jargon : qui est concerné, de combien, pourquoi.
- [ ] La dernière version du notebook est enregistrée dans GitHub.